# Пространства имён

## Цели лекции

После этой лекции вы сможете:
- применять правило **LEGB** для разрешения имён и понимать области видимости;
- осознанно использовать `global` и `nonlocal`;
- писать замыкания и декораторы;
- раскладывать код по модулям и пакетам и управлять тем, что видно снаружи (`__all__`);
- отделять точку входа от переиспользуемого кода через `if __name__ == "__main__"`;
- разбирать ошибки импорта, включая циклические, и запускать модуль через `-m`.

### Основы областей видимости (scopes) в Python
- **Имена** в программе существуют в пространстве имён (namespace).  
- **Область видимости (scope)** определяется местом присваивания имени в коде.  
- Все имена создаются в момент присваивания (объявления заранее нет).  
- Функции добавляют новый уровень пространства имён, чтобы избежать конфликтов переменных.  
- Это называется **лексическая область видимости**: она зависит только от расположения кода, а не от того, кто кого вызывает.  
- Переменные с одинаковыми именами внутри и снаружи функции – разные переменные. 


In [1]:
X = 99  # глобальная переменная

def func():
    X = 88  # локальная переменная, с глобальной X не связана
    print("Внутри функции X =", X)

func()
print("Снаружи функции X =", X)

Внутри функции X = 88
Снаружи функции X = 99


### Вызывающая функция не передаёт свою область видимости

Функция ищет внешние имена там, где она **определена**.
Локальные переменные того, кто её вызвал, в этот поиск не входят.

Значение из вызывающей функции передают аргументом.


In [1]:
location = "модуль"

def show_location():
    return location

def caller():
    location = "вызывающая функция"
    return show_location()

print(caller())


модуль


### Тонкости областей видимости
1. В интерактивной оболочке (`REPL`) всё работает как в модуле: переменные на верхнем уровне считаются глобальными.  
2. Любое присваивание делает имя локальным или глобальным в зависимости от места. Это относится к `=`, `import`, `def`, `class`, аргументам функций и циклам `for`.  
3. Изменение объекта **не меняет область видимости имени**. Это то же различие
   присваивания имени и изменения объекта, что при передаче аргументов в лекции 5:  
   - `L = X` внутри функции – присваивание: создаёт локальную переменную `L`;  
   - `L.append(X)` – изменение объекта: меняет список, на который указывает глобальная `L`, и не требует объявления `global`.  

### Локальное имя определяется по всему телу функции

Если внутри функции есть присваивание имени, без `global` или `nonlocal`
оно считается локальным **во всём теле**, даже если эта ветка не выполнится.

Чтение такого имени до получения значения вызывает `UnboundLocalError`.
Python не продолжает поиск одноимённой глобальной переменной.


In [2]:
status = "глобальное значение"

def choose_status(reset):
    if reset:
        status = "новое значение"
    return status

print(choose_status(True))
try:
    print(choose_status(False))
except UnboundLocalError as error:
    print("UnboundLocalError:", error)


новое значение
UnboundLocalError: cannot access local variable 'status' where it is not associated with a value


### Правило LEGB для разрешения имён
- Внутри функции (`def` или `lambda`) присваивания по умолчанию создают **локальные имена**.  
- Поиск имён при обращении к ним без точки (`name`) идёт в порядке **LEGB**:  
  1. **L (Local)** – локальная область текущей функции.  
  2. **E (Enclosing)** – локальные области всех внешних функций (если есть).  
  3. **G (Global)** – глобальная область текущего модуля.  
  4. **B (Built-in)** – встроенные имена Python (`len`, `open`, `range` и др.).  
- Поиск останавливается на первом найденном совпадении.  
- В коде вне функций поиск проходит только две последние ступени, G и B. Ступени L и E добавляются, когда имя используется внутри функции.  
- **Каждый вызов функции создаёт своё локальное пространство имён.** На этом держатся рекурсия и замыкания: у каждого вызова собственный набор локальных переменных.  
- На верхнем уровне файла локальная и глобальная области совпадают.  


Схема LEGB

In [2]:
# Глобальная переменная
X = 99  

def outer():
    X = 88  # Enclosing переменная

    def inner():
        X = 77  # Local переменная
        print("Local:", X)  # Берётся из inner

    inner()
    print("Enclosing:", X)  # Берётся из outer

outer()
print("Global:", X)  # Берётся из модуля
print("Built-in:", len([1, 2, 3]))  # Пример встроенной функции

Local: 77
Enclosing: 88
Global: 99
Built-in: 3


### Два замыкания могут разделять одну переменную

**Замыкание** – вложенная функция, которая использует переменные внешней функции
и сохраняет к ним доступ после того, как внешняя функция завершилась.
Так был устроен `make_multiplier` из лекции 5: `double` помнит `factor = 2`,
хотя вызов `make_multiplier(2)` уже закончился.

Функции, созданные одним вызовом внешней функции, могут читать и менять
общую переменную этого вызова.

Для чтения объявление не нужно. Для переприсваивания используем `nonlocal`:
оно обращается к уже существующему имени в ближайшей внешней функции.


In [3]:
def make_setting(initial):
    value = initial
    def read():
        return value
    def write(new_value):
        nonlocal value
        value = new_value
    return read, write

read_theme, set_theme = make_setting("светлая")
print(read_theme())
set_theme("тёмная")
print(read_theme())


светлая
тёмная


### Разные вызовы создают независимое состояние

Вызовем `make_setting` ещё раз. Каждый вызов создаёт **новую** переменную `value`
и новую пару функций `read`/`write`, которые работают только с ней.

Поэтому `set_language` меняет язык, но не тему: у функций темы своя `value`.
Общая переменная есть только у двух функций из **одного** вызова `make_setting`.


In [4]:
read_language, set_language = make_setting("ru")
set_language("en")
print("Язык:", read_language())
print("Тема:", read_theme())

set_theme("системная")
print("Язык после смены темы:", read_language())


Язык: en
Тема: тёмная
Язык после смены темы: en


### Дополнительные области видимости в Python
- Помимо стандартных областей LEGB, отдельно стоит запомнить три случая:  
  1. **Переменные в comprehensions** (`[x for x in I]`): имя `x` локально только для comprehension и не конфликтует с внешними переменными. В обычных циклах `for` переменная цикла остаётся доступной снаружи.  
  2. **Имя после `except E as X`**: своей области блок `except` **не создаёт** – `X` принадлежит окружающей области. Python лишь удаляет это имя при выходе из блока, поэтому прежнее значение `X`, если оно было, не восстановится.  
  3. **Присваивания внутри lambda с :=**: создают локальные переменные внутри выражения `lambda`.  
- Своя область здесь есть у comprehension и у `lambda` (это тоже функция): их переменные снаружи не видны. Правило LEGB они не меняют.  
- **class** тоже создаёт своё пространство имён, но для методов оно не становится enclosing-областью. Подробно – в лекции про классы.  

In [3]:
# Comprehension: переменная x локальна
squares = [x*x for x in range(3)]
try:
    print(x)  # Ошибка: x не определён
except NameError:
    print("x не доступен вне comprehension")

x не доступен вне comprehension


In [4]:
# Обычный for: переменная сохраняется
for y in range(3):
    pass
print("Переменная из for:", y)

Переменная из for: 2


In [20]:
# Переменная в except-блоке
try:
    1 / 0
except ZeroDivisionError as e:
    print("Ошибка внутри except:", e)
try:
    print(e)  # Ошибка: e удалён
except NameError:
    print("e не доступен вне except")

# Своей области у блока нет: прежнее значение того же имени не вернётся
e = "было до try"
try:
    1 / 0
except ZeroDivisionError as e:
    pass
try:
    print(e)
except NameError:
    print("прежнее значение e не восстановилось")

Ошибка внутри except: division by zero
e не доступен вне except
прежнее значение e не восстановилось


### Встроенная область видимости
- Встроенная область (`Built-in scope`) реализована модулем **`builtins`**.  
- Все встроенные функции и исключения – их около полутора сотен – доступны автоматически по правилу **LEGB** (уровень `B`).  
- Импорт `builtins` позволяет просматривать список встроенных имён через `dir(builtins)`.  
- Локальные и глобальные переменные могут **затенять** (shadow) встроенные имена.  
- Обычно затенение случайно: имя вроде `list`, `str` или `open` заняли под переменную, и в этой области встроенная функция больше недоступна (пример ниже).  
- Специальные имена `None`, `True`, `False` зарезервированы и не могут быть изменены.  
- Чтобы восстановить встроенное имя после случайного затенения, можно использовать `del name` или перезапустить сессию.  


In [7]:
import builtins

# Встроенная функция доступна по двум маршрутам
print(zip is builtins.zip)  # True

# Опасность затенения
def hider():
    open = "text"  # локальная переменная скрывает встроенный open
    open("data.txt")  

hider()

True


TypeError: 'str' object is not callable

In [11]:
# Затенение на глобальном уровне
open = 99
print("Затенённый open:", open)

# Удалим имя, чтобы вернуть встроенную функцию
del open
print("Восстановленный open:", open)
print("Снова встроенный:", open is builtins.open)

Затенённый open: 99
Восстановленный open: <built-in function open>
Снова встроенный: True


### Проектирование программ: минимизация глобальных переменных
- Функции должны общаться через **аргументы и возвращаемые значения**, а не через глобальные переменные.  
- Присваивание внутри функции по умолчанию создаёт локальное имя, поэтому, чтобы **присвоить** глобальному имени новое значение, нужен `global`. Изменению объекта (`L.append(...)`) он не нужен – см. «Тонкости областей видимости» выше.  
- Минусы глобальных переменных:  
  - значение зависит от порядка вызова функций;  
  - код труднее понимать и сопровождать;  
  - сильная зависимость функций друг от друга.  
- Глобальные переменные можно использовать для хранения состояния, но это крайний случай.  
- В многопоточной программе глобальная переменная общая для всех потоков: без блокировки одновременные изменения могут теряться.  


In [9]:
# Плохой дизайн: функции зависят от общей переменной X
X = 99

def func1():
    global X
    X = 88

def func2():
    global X
    X = 77

func1()
print(X)  # 88
func2()
print(X)  # 77 (значение зависит от порядка вызова)

88
77


### Другие способы доступа к глобальным переменным
- Глобальные переменные = атрибуты модуля.  
- Способы **присвоить** глобальному имени новое значение:  
  1. `global` внутри функции – для имени своего же модуля это предпочтительный вариант.  
  2. Импорт самого модуля и изменение его атрибутов: в файле `thismod.py` – `import thismod`, затем `thismod.var += 1`.  
  3. Словарь `sys.modules`, где хранятся все загруженные модули: текущий лежит в нём под именем `__name__`.  
- Работают все три, но обычно берут `global`: он короче и явно показывает намерение. Основная рекомендация при этом не меняется – передавать данные аргументами и возвращать результат.  

In [13]:
import sys

var = 99  # глобальная переменная = атрибут текущего модуля


def bump_with_global():
    global var
    var += 1


def bump_with_module():
    module = sys.modules[__name__]   # текущий модуль из словаря загруженных
    module.var += 1                  # меняем его атрибут – это та же var


bump_with_global()
bump_with_module()
print(var)                           # 101: обе функции изменили одну переменную

101


## Модули и пакеты

**Модуль** – это просто `.py`-файл; каждый модуль – отдельное пространство имён верхнего
уровня. `import` находит модуль, **выполняет его один раз** и связывает с именем.
Формы импорта:
- `import math` – весь модуль, доступ через `math.sqrt`;
- `from math import pi, sqrt` – выборочно нужные имена;
- `import numpy as np` – с псевдонимом.

In [11]:
import math                    # импорт модуля целиком
from math import pi, sqrt      # выборочный импорт имён
import statistics as st        # импорт с псевдонимом

print(math.factorial(5))       # обращение через имя модуля
print(pi, sqrt(16))            # напрямую
print(st.mean([1, 2, 3, 4]))   # через псевдоним

120
3.141592653589793 4.0
2.5


### `from`-импорт не видит нового значения атрибута

Два файла лежат рядом. Запускаем `main.py` в новом процессе Python.

```python
# settings.py
limit = 10
```

```python
# main.py
import settings
from settings import limit

settings.limit = 20
print(settings.limit, limit)  # 20 10
```

`from settings import limit` создаёт в `main.py` своё имя `limit` и связывает
его с тем объектом, на который в момент импорта указывал `settings.limit`.
Присваивание `settings.limit = 20` направляет атрибут модуля на новый объект,
а имя `limit` в `main.py` по-прежнему указывает на старый – `10`.

### Присваивание импортированному имени не меняет модуль

Продолжим **тот же `main.py`**:

```python
limit = 30
print(settings.limit, limit)  # 20 30
```

Теперь изменилось только имя в `main.py`. Актуальное значение
атрибута читают через сам модуль: `settings.limit`.

При этом `from` не копирует объект. Если в модуле лежит список и его
**изменить** на месте, изменение видно через оба имени:

```python
# в settings.py добавлено: admins = ["anna"]
from settings import admins
admins.append("boris")
print(settings.admins)  # ['anna', 'boris'] – это тот же список
```

То же различие присваивания имени и изменения объекта, что при передаче
аргументов в лекции 5.

### Когда свой файл перекрывает библиотеку

Эту ловушку называли в «Частых ошибках» лекции 1. Разберём, как она выглядит.
Пусть рядом лежат `main.py` и собственный `random.py`:

```python
# random.py
label = "Мои случайные числа"
```

```python
# main.py
import random
print(random.choice(["Анна", "Борис"]))
# AttributeError: module 'random' has no attribute 'choice'
```

При обычном запуске `python main.py` каталог скрипта стоит в `sys.path`
раньше каталогов стандартной библиотеки. Поэтому найден наш `random.py`.
Встроенные в интерпретатор модули ищутся отдельно; это правило не для всех имён.

### Как проверить, какой модуль загрузился

В предыдущем `main.py` **перед проблемным вызовом** добавим:

```python
print(random.__file__)  # путь к загруженному random.py
```

Если путь ведёт в свой проект, найден источник конфликта.

1. Переименовать свой файл, например в `random_tools.py`.
2. Обновить импорты, которые обращались именно к своему модулю.
3. Запустить программу заново. В Jupyter – перезапустить ядро.

Почему одного повторного `import` недостаточно, разбирали в лекции 1.
`__file__` полезен для файловых модулей; у некоторых встроенных модулей его нет.

### Публичные и вспомогательные имена модуля

Публичные имена – те, которые автор предлагает использовать снаружи.
Начальное `_` обозначает внутреннюю деталь, которая может измениться, –
это соглашение об именах из лекции 4.

```python
# name_tools.py
VERSION = 1

def _normalize(text):
    return text.strip().title()

def format_name(text):
    return "Участник: " + _normalize(text)
```

Пользователь вызывает `format_name`, а вспомогательная `_normalize` остаётся
деталью реализации. Подчёркивание – соглашение, не запрет доступа:
`name_tools._normalize(...)` технически допустим.

### Что именно регулирует `__all__`

Добавим в `name_tools.py` строку `__all__ = ["format_name"]`.
Теперь импорт со звёздочкой выберет только это имя:

```python
# main.py
from name_tools import *
print(format_name("  анна  "))  # Участник: Анна
print("VERSION" in globals())   # False: имя не импортировано

import name_tools
print(name_tools.VERSION)               # 1
print(name_tools._normalize("  борис ")) # Борис
```

Без `__all__` звёздочка забрала бы все имена, которые не начинаются с `_`:
`VERSION` и `format_name`, но не `_normalize`.
`__all__` управляет только `import *`, а не доступом через модуль или явным
`from name_tools import VERSION`. В обычном коде предпочтительнее явные импорты;
`globals()` здесь позволяет проверить наличие имени в текущем модуле.

### Главный блок: `if __name__ == "__main__"`

У каждого модуля есть имя в переменной `__name__`. Если файл **запущен напрямую**
(`python file.py`), то `__name__ == "__main__"`. Если же он **импортирован**, `__name__`
равен имени модуля. Отсюда идиома: код внутри `if __name__ == "__main__":` выполняется
только при прямом запуске, но **не** при импорте – удобно отделять «точку входа» от
переиспользуемого кода.

In [12]:
# __name__ при прямом запуске и при импорте
import sys
import tempfile
from pathlib import Path

print("Здесь __name__ =", __name__)   # запущено напрямую -> __main__

# Создадим модуль на лету и импортируем – при ИМПОРТЕ __name__ другой
with tempfile.TemporaryDirectory() as d:
    (Path(d) / "greeter.py").write_text(
        'print("  при импорте greeter.__name__ =", __name__)\n'
        'def hello():\n    return "привет из greeter"\n'
        'if __name__ == "__main__":\n'
        '    print("  этот блок НЕ сработает при импорте")\n',
        encoding="utf-8")
    sys.path.insert(0, d)
    import greeter                     # импорт выполняет код модуля один раз
    print("  greeter.hello() ->", greeter.hello())
    sys.path.remove(d)

Здесь __name__ = __main__
  при импорте greeter.__name__ = greeter
  greeter.hello() -> привет из greeter


### Пакеты и структура проекта

**Пакет** – это папка с модулями. Обычный пакет помечается файлом `__init__.py`.
Модуль внутри пакета называют **подмодулем** и импортируют по пути через точку: `mypackage.utils` –
это файл `mypackage/utils.py`, точка заменяет разделитель папок.

Папка без `__init__.py` тоже импортируется – как *пакет пространства имён*
(namespace package), части которого могут лежать в разных папках. Своему
проекту такая гибкость не нужна, поэтому `__init__.py` лучше создавать:
так сразу видно, что папка – пакет.

```
myproject/
    mypackage/
        __init__.py        # делает папку пакетом
        core.py            # def run(): ...
        utils.py
    main.py                # точка входа
```

```python
# main.py
from mypackage import core        # подмодуль core из пакета mypackage
from mypackage.utils import clean # функция clean из подмодуля utils

core.run()
```

Чаще всего `__init__.py` пустой: его задача – пометить папку как пакет. Но это
обычный модуль: он выполняется один раз, при первом импорте пакета или любого
его подмодуля, а созданные в нём имена становятся атрибутами пакета. Поэтому
в нём пишут короткий «фасад»:

```python
# mypackage/__init__.py
from .utils import clean   # теперь можно писать from mypackage import clean
__version__ = "1.0"
```

Так устроен `requests`: `requests.get` определена в `requests/api.py`. Тяжёлый
код в `__init__.py` не кладут: он выполнится при любом импорте пакета.

Проект стоит разбивать на модули по смыслу – так код проще читать,
тестировать и переиспользовать.

### Точка в импорте означает текущий пакет

В структуре `myproject/mypackage/` с предыдущего экрана:

```python
# mypackage/utils.py
def clean(text):
    return text.strip().lower()
```

```python
# mypackage/core.py
from .utils import clean

def run():
    print(clean("  Анна  "))

if __name__ == "__main__":
    run()
```

`.utils` – модуль внутри `mypackage`. Точка относится к пакету,
а не к текущей папке терминала. Абсолютный вариант: `from mypackage.utils import clean`.

### Два способа запуска: какая папка попадает в `sys.path`

Имя модуля без точки (`import utils`, `from mypackage.utils import clean`)
Python ищет в папках списка `sys.path`. Первой в нём стоит папка, которую
добавляет способ запуска. Обе команды выполняем из `myproject/`:

```
myproject/          ← python -m mypackage.core:  sys.path[0] = myproject/
    main.py
    mypackage/      ← python mypackage/core.py:  sys.path[0] = mypackage/
        __init__.py
        core.py
        utils.py
```

- **`python -m mypackage.core`**: `-m` получает **имя модуля через точку**, без `.py`.
  Из `myproject/` виден пакет `mypackage`: Python импортирует его и выполняет `core`
  как его часть. Модуль знает свой пакет, и `.utils` означает `mypackage.utils`.
- **`python mypackage/core.py`**: Python знает только файл и кладёт в `sys.path`
  его папку. Соседи `core.py` видны как отдельные модули `utils`, `core`. Пакета нет:
  на `__init__.py` Python не смотрит, и точке в `from .utils` не от чего отсчитываться.

```bash
python -m mypackage.core
# анна

python mypackage/core.py
# ImportError: attempted relative import with no known parent package
```

`-m` ищет пакет в текущей папке, поэтому команду запускают из папки, где лежит
папка пакета. Изнутри `mypackage/` получится `No module named 'mypackage'`.

### Какой импорт выбрать

Как импортировать `clean` в `core.py`, чтобы модуль работал при разных запусках
(всё выполняется из `myproject/`):

| Импорт в `core.py` | `python mypackage/core.py` | `python -m mypackage.core` | `python main.py` |
| --- | --- | --- | --- |
| `from utils import clean` | работает | `No module named 'utils'` | `No module named 'utils'` |
| `from .utils import clean` | `no known parent package` | работает | работает |
| `from mypackage.utils import clean` | `No module named 'mypackage'` | работает | работает |

`python main.py` – обычная точка входа: `main.py` лежит вне пакета и делает
`from mypackage import core`. `sys.path` задаёт запущенный файл, а не тот, где
написан импорт: первой в нём стоит папка `main.py`, то есть `myproject/`, а `utils.py`
лежит глубже, в `mypackage/`. Поэтому первая строка таблицы работает только при
запуске `core.py` по пути, и любой импорт `core` извне её ломает.

Правила:

- соседей **внутри пакета** импортируют от имени пакета: относительно
  (`from .utils import clean`) – коротко и не зависит от названия пакета – или
  полностью (`from mypackage.utils import clean`) – так же, как в `main.py` и тестах;
- импорт без пакета (`from utils import ...`) – только для одиночных скриптов,
  которые лежат рядом и ни в какой пакет не входят;
- модуль пакета запускают через `-m` из корня проекта, а `main.py` вне пакета –
  обычным `python main.py`.

### Подпакеты: на уровень глубже

Внутри пакета может лежать ещё один пакет – подпакет со своим `__init__.py`:

```
myproject/
    main.py
    mypackage/
        __init__.py
        utils.py
        reports/          ← подпакет mypackage.reports
            __init__.py
            formats.py
            summary.py
```

В относительном импорте число точек – на сколько уровней подняться:

```python
# mypackage/reports/summary.py
from .formats import as_title   # сосед по подпакету: mypackage.reports.formats
from ..utils import clean       # уровнем выше: mypackage.utils
```

Полный путь всегда считается от верхнего пакета: `from mypackage.utils import clean`.
Запускают по-прежнему из корня проекта, только точек в имени больше:
`python -m mypackage.reports.summary`.

- **Импорт без пакета здесь не работает ни при каком запуске.** Даже при запуске
  по пути в `sys.path` попадает только папка `reports/`, а `utils.py` лежит выше.
- **Выше верхнего пакета точками не подняться.** `from ...utils` в `summary.py`
  даёт `attempted relative import beyond top-level package`. Та же ошибка будет,
  если запустить `python -m reports.summary` изнутри `mypackage/`: верхним пакетом
  тогда станет `reports`.

### Циклический импорт: модули зависят друг от друга

Три отдельных файла в одной папке; запускаем `python main.py`:

```python
# main.py
from catalog import show
print(show(["Хлеб", "Молоко"]))

# catalog.py
from reports import render
TITLE = "Товары"

def show(items):
    return render(items)

# reports.py
from catalog import TITLE

def render(items):
    return TITLE + ": " + ", ".join(items)
```

Зависимости: `main → catalog → reports → catalog`.
Импорт завершается ошибкой ещё до вызова `show()`.

### Что значит «частично инициализированный модуль»

В предыдущем примере Python проходит такую цепочку:

1. Начинает выполнять `catalog.py`; объект модуля уже помещён в `sys.modules`.
2. На первой строке переключается на выполнение `reports.py`.
3. В `reports.py` встречает `from catalog import TITLE`.
4. Находит уже начатый `catalog`, но до строки `TITLE = ...` ещё не дошёл.

**Что напишет Python.** Сообщение начинается с
`ImportError: cannot import name 'TITLE'`, продолжение зависит от версии:

- до 3.12 включительно – `from partially initialized module 'catalog' (most likely due to a circular import)`;
- в 3.13 – совет переименовать `catalog.py`, как будто файл перекрывает
  библиотеку. Здесь совет ложный: причина в цикле.

**Как найти причину.** В traceback ищут переход из одного своего модуля
в другой и обратный импорт.

**Когда цикл не мешает.** Ошибку даёт не сам цикл, а обращение к атрибуту,
который ещё не создан. Поэтому `import catalog` вместо `from catalog import TITLE`
не поможет, если сразу после импорта читать `catalog.TITLE`.

### Разрываем цикл: выносим общее в отдельный модуль

`TITLE` описывает оформление отчёта; ему не нужно зависеть от каталога.
Оставим `main.py` прежним, а остальные файлы запишем так:

```python
# report_config.py
TITLE = "Товары"

# reports.py
from report_config import TITLE

def render(items):
    return TITLE + ": " + ", ".join(items)

# catalog.py
from reports import render

def show(items):
    return render(items)
```

Теперь зависимости идут в одну сторону: `catalog → reports → report_config`.

Отдельный модуль подходит для постоянного значения. Если заголовок меняется
от вызова к вызову, его передают аргументом: `render(items, title)`. Иначе
общий модуль превратится в глобальную переменную, которую меняют из разных
мест, – с теми же проблемами, что разбирались выше.

### Отложенный импорт: когда он помогает

Другой способ – оставить три **исходных** файла и изменить только `catalog.py`:

```python
# catalog.py – было
from reports import render
TITLE = "Товары"

def show(items):
    return render(items)
```

```python
# catalog.py – стало
TITLE = "Товары"

def show(items):
    from reports import render   # выполнится только при вызове show()
    return render(items)
```

Импорт внутри функции называют **локальным** или отложенным. Теперь `python main.py`:

1. `main.py` импортирует `catalog`, и тот выполняется до конца: создаётся `TITLE`,
   определяется `show`. Строка импорта внутри `show` пока не выполняется.
2. `main.py` вызывает `show()`, и только теперь выполняется `from reports import render`.
3. `reports.py` читает `from catalog import TITLE`: `catalog` уже загружен
   целиком, `TITLE` существует. Ошибки нет.

### Отложенный импорт не убирает зависимость

Зависимость `catalog → reports` осталась, изменилось только **время** импорта.
Поэтому важно, когда вызывают `show()`, а не из какого модуля:

- **после** загрузки `catalog` – безопасно, как в `main.py` или в любом другом
  модуле, который сначала импортировал `catalog`;
- **во время** загрузки `catalog` – ошибка вернётся, если `TITLE` к этому моменту
  ещё не создан: например, если на верхнем уровне `catalog.py` вызвать `show()`
  выше строки `TITLE = ...`.

Для общих данных надёжнее отдельный модуль или передача аргументом. Локальный
импорт оставляют для случаев, когда отложенная загрузка нужна сознательно.

## Декораторы функций
- **Декоратор** получает функцию и возвращает объект, который будет доступен под её именем. Часто это сама функция или функция-обёртка.
- Синтаксис `@decorator` над `def func(): ...` – это **синтаксический сахар**: он полностью эквивалентен `func = decorator(func)`.
- Декораторы – практическое применение **замыканий** (enclosing-переменных из правила LEGB, которое мы разобрали выше): обёртка «помнит» исходную функцию через enclosing-область.

In [13]:
def logged(func):
    def wrapper(*args, **kwargs):
        print(f"Вызов {func.__name__} с args={args}, kwargs={kwargs}")
        result = func(*args, **kwargs)
        print(f"{func.__name__} вернула {result!r}")
        return result
    return wrapper

def add(a, b):
    return a + b

# Без синтаксиса @: ровно то же самое, что делает декоратор
add = logged(add)
add(2, 3)

print("---")

# То же самое через @decorator – просто синтаксический сахар
@logged
def multiply(a, b):
    return a * b

multiply(4, 5)

Вызов add с args=(2, 3), kwargs={}
add вернула 5
---
Вызов multiply с args=(4, 5), kwargs={}
multiply вернула 20


### Как декоратор оборачивает функцию: `*args/**kwargs` и `functools.wraps`
- Обёртка (`wrapper`) должна принимать `*args, **kwargs`, чтобы декоратор одинаково работал с любой оборачиваемой функцией – независимо от её сигнатуры.
- Без дополнительных мер обёртка **подменяет** `__name__`, `__doc__` и другие атрибуты исходной функции – снаружи будет видна не `slow_sum`, а `wrapper`.
- `functools.wraps(func)` – декоратор для декораторов: копирует метаданные (`__name__`, `__doc__` и др.) из оригинальной функции на обёртку.

In [14]:
import functools
import time

def timer(func):
    @functools.wraps(func)  # сохраняет __name__, __doc__ оригинальной функции
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} выполнилась за {elapsed:.6f} сек")
        return result
    return wrapper

@timer
def slow_sum(n):
    """Считает сумму чисел от 0 до n-1."""
    return sum(range(n))

slow_sum(1_000_000)
print("Имя функции:", slow_sum.__name__)   # slow_sum, а не wrapper
print("Докстринг:", slow_sum.__doc__)

slow_sum выполнилась за 0.011093 сек
Имя функции: slow_sum
Докстринг: Считает сумму чисел от 0 до n-1.


### Что происходит с таймером, если функция падает

Применим предыдущий `timer` к `int`. В обёртке печать времени стоит **после**
строки `result = func(*args, **kwargs)`. Когда `int("три")` выбрасывает
`ValueError`, обёртка прерывается на этой строке: исключение уходит
к вызывающему коду, а до печати дело не доходит.

Нужно, чтобы время печаталось при любом выходе из обёртки – и с результатом,
и с исключением.

In [1]:
timed_int = timer(int)
try:
    timed_int("три")
except ValueError:
    print("Ошибка дошла до вызывающего кода, время не напечатано")

Ошибка дошла до вызывающего кода, время не напечатано


### Замер времени и при результате, и при исключении

Перенесём печать в знакомый по лекции 4 блок `finally`.
`return` остаётся в `try`: результат возвращается, а исключение проходит наружу.
В `finally` не добавляем `return`: он подавил бы исключение – это ловушка
из «Частых ошибок» лекции 4.

В выводе две строки времени: первая – от неудачного вызова, вторая – от удачного.
`int` – это `func.__name__`, а нули – потому что преобразование занимает
меньше микросекунды.

In [2]:
def timer_always(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            elapsed = time.perf_counter() - start
            print(f"{func.__name__}: {elapsed:.6f} сек")
    return wrapper

timed_int = timer_always(int)
try:
    timed_int("три")                   # finally печатает время, ValueError уходит наружу
except ValueError:
    print("Ошибка по-прежнему дошла до вызывающего кода")
print("Результат:", timed_int("12"))   # сначала вызов (и печать времени), потом print

int: 0.000001 сек
Ошибка по-прежнему дошла до вызывающего кода
int: 0.000000 сек
Результат: 12


### Декоратор применяется при определении функции

Когда Python доходит до декорированного `def`, он создаёт функцию
и сразу передаёт её декоратору.

Внешний код декоратора выполняется сейчас. Тело возвращённой обёртки
начнёт выполняться при последующем вызове.


In [5]:
from functools import wraps

def announce_definition(fn):
    print("Декорируем:", fn.__name__)    # выполняется сразу, при определении welcome
    @wraps(fn)
    def wrapped(*args, **kwargs):
        print("Вызов обёртки")            # выполнится только при вызове welcome(...)
        return fn(*args, **kwargs)
    return wrapped

@announce_definition                     # то же, что welcome = announce_definition(welcome)
def welcome(name):
    print("Тело функции")
    return f"Здравствуйте, {name}"

print("Определение завершено")           # welcome ещё не вызывали: обёртка и тело молчат


Декорируем: welcome
Определение завершено


### Вызов обёртки не повторяет декорирование

Теперь дважды вызовем `welcome`. Каждый раз выполнятся обёртка
и исходная функция.

Сообщение «Декорируем» не появится: декоратор уже применён.
Если заново выполнить ячейку с определением, декорирование повторится.


In [6]:
# welcome теперь – обёртка wrapped. При каждом вызове:
# 1) wrapped печатает «Вызов обёртки»; 2) вызывает исходную welcome – «Тело функции»;
# 3) исходная welcome возвращает строку, и её печатает внешний print
print(welcome("Анна"))
print(welcome("Борис"))


Вызов обёртки
Тело функции
Здравствуйте, Анна
Вызов обёртки
Тело функции
Здравствуйте, Борис


### Регистрация функции без обёртки

Декоратор может сохранить функцию в реестре и вернуть **ту же функцию**.
Зачем: программа получает команду строкой – из терминала, из сообщения
в чате, из адреса в браузере – и по этой строке находит обработчик. Реестр
заполняется сам, пока Python выполняет `def` с декоратором, и словарь
«имя → функция» не нужно вести вручную.

Регистрация происходит при определении. Вызов через реестр выполняет
исходную функцию, дополнительного вызова обёртки здесь нет.


In [7]:
commands = {}

def register(fn):
    commands[fn.__name__] = fn
    return fn

@register
def hello():
    return "Привет!"

print("Команды:", sorted(commands))
print("Та же функция:", commands["hello"] is hello)
print(commands["hello"]())


Команды: ['hello']
Та же функция: True
Привет!


### Декораторы с аргументами (параметризованные)
- Если декоратору нужен собственный параметр (например, `@repeat(n=3)`), добавляется ещё один уровень вложенности функций.
- Внешняя функция (`repeat`) принимает параметр декоратора и возвращает **сам декоратор** (`decorator`), который уже принимает оборачиваемую функцию.
- Запись `@repeat(n=3)` над `def cheer` равна `cheer = repeat(n=3)(cheer)`: сначала вызывается `repeat` и возвращает декоратор, затем декоратор применяется к `cheer`.

In [23]:
import functools

def repeat(n):
    """Фабрика декораторов: возвращает декоратор с зафиксированным параметром n."""
    def decorator(func):                    # func – декорируемая функция, здесь cheer
        @functools.wraps(func)
        def wrapper(*args, **kwargs):       # args – аргументы вызова, здесь ("команда",)
            result = None
            for _ in range(n):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(n=3)            # repeat(n=3) возвращает decorator, decorator(cheer) – wrapper
def cheer(team):
    print(f"Вперёд, {team}!")

cheer("команда")        # на самом деле вызывается wrapper("команда")

Вперёд, команда!
Вперёд, команда!
Вперёд, команда!


### Несколько декораторов подряд
- Декораторы можно накладывать друг на друга – при определении они применяются **снизу вверх**.
- `@bold` над `@italic` над `def heading` эквивалентно `heading = bold(italic(heading))`.
- Тело `heading` при этом не выполняется. При **вызове** порядок обратный: сначала обёртка `bold`, внутри неё обёртка `italic`, и только затем тело `heading`. Результат возвращается наружу в обратном порядке.

In [24]:
def bold(func):
    def wrapper(*a, **kw):
        return f"<b>{func(*a, **kw)}</b>"
    return wrapper

def italic(func):
    def wrapper(*a, **kw):
        return f"<i>{func(*a, **kw)}</i>"
    return wrapper

@bold
@italic
def heading(text):
    return text.upper()

print(heading("итоги"))
# При определении: heading = bold(italic(heading)) – снизу вверх.
# При вызове: обёртка bold → обёртка italic → тело heading ("ИТОГИ"),
# затем italic добавляет <i>, а bold – <b>

<b><i>ИТОГИ</i></b>


## Частые ошибки

**Позднее связывание в замыканиях.** Замыкание запоминает **переменную**, а не её значение
на момент создания. Все лямбды из цикла увидят последнее значение `i`.

Исправление опирается на правило из «Частых ошибок» лекции 5: значение по умолчанию
вычисляется один раз, при определении функции. Там это было ловушкой, здесь работает на нас.

In [17]:
funcs = [lambda: i for i in range(3)]
print([f() for f in funcs])          # [2, 2, 2] – все берут последнее i!

# Правильно: зафиксировать значение через аргумент по умолчанию.
# n=i вычисляется при создании каждой lambda: у первой n=0, у второй 1, у третьей 2.
# f() вызывается без аргументов, поэтому n берёт значение по умолчанию
funcs_ok = [lambda n=i: n for i in range(3)]
print([f() for f in funcs_ok])       # [0, 1, 2]

# В чужом коде то же самое часто пишут как lambda i=i: i –
# там параметр просто назван так же, как переменная цикла

[2, 2, 2]
[0, 1, 2]


### Забытый `nonlocal` не даёт ошибки

Без объявления присваивание во вложенной функции создаёт **новое локальное имя**.
Внешняя переменная остаётся прежней, и программа молча работает со старым
значением. С `global` на уровне модуля происходит то же самое.

Исправление – `nonlocal enabled` в начале `turn_on`. Если же имя сначала читается,
как в `n += 1`, Python сразу выбросит `UnboundLocalError`, и ошибку заметно.

In [26]:
def make_switch():
    enabled = False

    def turn_on():
        enabled = True           # без nonlocal: новое локальное имя внутри turn_on

    def is_on():
        return enabled

    return turn_on, is_on


turn_on, is_on = make_switch()
turn_on()
print("Включено:", is_on())      # False: присваивание не дошло до внешней переменной

Включено: False


## Мини-упражнения


### 1. `make_counter()`

Верните функцию-счётчик: при каждом вызове увеличивает и возвращает внутреннее число
(замыкание + `nonlocal`).

In [18]:
def make_counter():
    n = 0
    def counter():
        nonlocal n
        n += 1
        return n
    return counter

c = make_counter()
print(c(), c(), c())

1 2 3


### 2. `@once`

Декоратор, который выполняет функцию только при первом вызове, а дальше
возвращает запомненный результат. Состояние нужно хранить в замыкании (`nonlocal`),
а метаданные – сохранить через `functools.wraps`.

In [19]:
import functools


def once(fn):
    done, saved = False, None

    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        nonlocal done, saved
        if not done:
            saved = fn(*args, **kwargs)
            done = True
        return saved

    return wrapper


@once
def load_config():
    print("читаем файл настроек")
    return {"debug": True}


print(load_config())    # первый вызов: тело выполняется, печатает «читаем файл настроек»
print(load_config())    # второй: тело не выполняется, print выводит сохранённый результат
print("имя сохранилось:", load_config.__name__)

читаем файл настроек
{'debug': True}
{'debug': True}
имя сохранилось: load_config


### 3. Регистрация без потери прежней команды

Напишите декоратор `register_unique` для словаря `unique_commands`:

- сохранить функцию по её имени и вернуть её без обёртки;
- если имя уже занято, выбросить `ValueError` и сохранить прежнюю запись.

Проверьте обычный вызов зарегистрированной функции и попытку зарегистрировать
другую функцию с таким же именем.


In [8]:
import pytest

unique_commands = {}
def register_unique(fn):
    if fn.__name__ in unique_commands:
        raise ValueError("Имя команды уже занято")
    unique_commands[fn.__name__] = fn
    return fn

@register_unique              # первая ping: имя свободно, функция попадает в реестр
def ping():
    return "pong"

# Вторая ping с тем же именем: декоратор выбрасывает ValueError раньше,
# чем имени ping присвоится новая функция. pytest.raises ловит ожидаемую ошибку
with pytest.raises(ValueError):
    @register_unique
    def ping():
        return "другой ответ"

# И реестр, и имя ping по-прежнему указывают на первую функцию
assert unique_commands["ping"] is ping
assert ping() == "pong"
print("Повтор отклонён, прежняя команда работает:", ping())


Повтор отклонён, прежняя команда работает: pong


### 4. Почините импорты

В папке `project/draw_app/` лежат пустой `__init__.py` и три файла:

```python
# random.py
TITLE = "Дежурный"

# utils.py
import random

def choose():
    return random.choice(["Анна", "Борис", "Вера"])

# core.py
from random import TITLE
from utils import choose

print(f"{TITLE}: {choose()}")
```

Из `project/` выполняют `python draw_app/core.py` и получают `AttributeError`.
Какой `random` загружен? Исправьте имя файла, импорты и команду запуска,
чтобы `core.py` работал как модуль пакета. Сохраните выбор случайного имени.

### Решение: исправляем проект и запускаем пакет

Что было сломано и как это исправить:

1. **Свой `random.py` перекрыл стандартный модуль.** `import random` в `utils.py`
   находил его, а в нём нет `choice` – отсюда `AttributeError`. Переименуем файл
   в `labels.py`, и `import random` снова найдёт стандартную библиотеку.
   Поэтому `utils.py` менять не нужно.
2. **`core.py` импортировал соседей как отдельные модули.** `from utils import ...`
   работает только при запуске файла по пути, когда папка `draw_app/` попадает
   в `sys.path`. Модулю пакета нужны импорты от имени пакета (см. «Какой импорт выбрать»):
   `from .labels import TITLE`,
   `from .utils import choose`. Печать – в главный блок `if __name__ == "__main__":`.
3. **Запуск по пути не знает о пакете.** Из папки над `draw_app/` запускаем
   **`python -m draw_app.core`**.

Ячейка ниже создаёт исправленные файлы во временной папке и запускает пакет
отдельным процессом; имя дежурного выбирается случайно.

In [30]:
import subprocess
import sys
import tempfile
from pathlib import Path

# Исправленный проект: ключи – пути файлов, значения – их текст.
# Свой random.py переименован в labels.py
files = {
    "draw_app/__init__.py": "",
    "draw_app/labels.py": 'TITLE = "Дежурный"\n',
    "draw_app/utils.py": """import random


def choose():
    return random.choice(["Анна", "Борис", "Вера"])
""",
    "draw_app/core.py": """from .labels import TITLE
from .utils import choose

if __name__ == "__main__":
    print(f"{TITLE}: {choose()}")
""",
}

with tempfile.TemporaryDirectory() as project:
    for name, text in files.items():
        path = Path(project) / name
        path.parent.mkdir(exist_ok=True)   # папка draw_app, если её ещё нет
        path.write_text(text, encoding="utf-8")
    # -X utf8 – дочерний Python пишет вывод в UTF-8: на Windows без этого
    # кириллица придёт в системной кодировке. check=True – выбросить исключение,
    # если процесс завершился с ошибкой; capture_output – собрать вывод в result.stdout
    result = subprocess.run(
        [sys.executable, "-X", "utf8", "-m", "draw_app.core"], cwd=project,
        capture_output=True, text=True, encoding="utf-8", check=True,
    )
print(result.stdout, end="")

Дежурный: Анна


### 5. Найдите две потери результата в декораторах

Оба декоратора ошибочны. Почему первый результат вообще нельзя вызвать,
а второй вызывается, но теряет число `7`? Найдите по одной пропущенной строке.

In [1]:
def missing_wrapper(fn):
    def wrapper(*args, **kwargs):
        return fn(*args, **kwargs)


def missing_result(fn):
    def wrapper(*args, **kwargs):
        fn(*args, **kwargs)
    return wrapper

first = missing_wrapper(lambda: 7)
second = missing_result(lambda: 7)
print("После первого декоратора:", first)
print("После вызова второй обёртки:", second())

После первого декоратора: None
После вызова второй обёртки: None


### Решение: два `return` выполняют разные задачи

- `missing_wrapper` не возвращает `wrapper`. Функция без `return` возвращает
  `None`, поэтому `first` – это `None`, а с `@missing_wrapper` имя функции
  тоже стало бы `None`.
- `missing_result` возвращает обёртку, но обёртка не возвращает результат `fn`:
  `7` вычисляется и теряется, вызов даёт `None`.

Ниже обе строки на своих местах.

In [2]:
from functools import wraps

def keep_result(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        return fn(*args, **kwargs)   # 1-й return: результат fn – тому, кто вызвал обёртку
    return wrapper                   # 2-й return: обёртка займёт имя функции

@keep_result
def triple(number):
    return number * 3

assert triple(number=4) == 12
assert triple.__name__ == "triple"
print(triple(3))

9


## Что почитать

- [Области видимости и пространства имён](https://docs.python.org/3/tutorial/classes.html#python-scopes-and-namespaces) –
  правило LEGB, `global` и `nonlocal`.
- [`functools.wraps`](https://docs.python.org/3/library/functools.html#functools.wraps) – что именно
  копируется из функции в обёртку.
- Л. Рамальо, «Python. К вершинам мастерства» – главы о замыканиях и декораторах.